In [1]:
!pip install sentence_transformers PyPDF2 faiss_cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 62.9 MB/s eta 0:00:00:00:0100:01


In [2]:
import transformers
import torch
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
import faiss
import numpy as np

In [3]:
from huggingface_hub import login
login()

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype = torch.float16, device_map = "auto")

def generate_text(prompt, max_length=100, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )

    return [
        tokenizer.decode(output, skip_special_tokens=True)
        for output in outputs
    ]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

The tokenizer you are loading from 'mistralai/Mistral-Nemo-Instruct-2407' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

# 1) Load PDF and extract text

In [7]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"
    return full_text

# 2) Parsing (Chunking)

In [8]:
def chunk_text(text, chunk_size = 500, overlap = 50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

# 3) Embedding

In [9]:
def embed_chunks(chunks, model_name = 'sentence-transformers/all-MiniLM-L6-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy = True)
    return model, embeddings

# 4) Indexing -> Vector Databse (FAISS)

In [10]:
def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim) # Calculate the similarity
    index.add(embeddings)
    return index

# 5) Search (Questions)

In [11]:
def search_index(query, model, index, chunks, k = 5):
    query_embedding = model.encode([query], convert_to_numpy = True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

# Upload a PDF

In [38]:
pdf_path = "/kaggle/input/datasets/omar1234321/python-pdf-simple/python_report_simple.pdf"
text = extract_text_from_pdf(pdf_path)
chunks = chunk_text(text, chunk_size = 50, overlap = 5)

model_embeddings, embeddings = embed_chunks(chunks)
index = create_faiss_index(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
question = "In what country and what kind of institution did Whitfield work when he created Python?"
top_chunks = search_index(question, model_embeddings, index, chunks, k = 3)

for i, chunk in enumerate(top_chunks, 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
As of early 2026, the latest major release is Python 3.13. Who Created It? Python was designed by the Dutch programmer Guido van Rossum. He began working on it in late 1989 at the CWI research institute in the Netherlands, during a Christmas holiday, as a personal project to keep

--- Chunk 2 ---
two of them. It is used in NASA space missions and in controlling robotic vehicles. The very first version of Python didn't even have lambda functions or map/filter. The Python Software Foundation (PSF) is a non-profit that officially oversees the language's development, and Python has consistently been one of the

--- Chunk 3 ---
Python The Programming Language: Origins, History & Uses Overview Python is a high-level, interpreted, general-purpose programming language. It's known for its readability and clean, simple syntax, which makes it one of the best languages for beginners to learn programming from scratch — while remaining powerful enough to build massive, complex


In [44]:
question = "In what country and what kind of institution did Whitfield work when he created Python?"
chunk = top_chunks[0]

prompt = f"Answer the next question: {question} by reading the following text:{chunk}"

In [47]:
answer = generate_text(prompt, max_length = 500)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [48]:
print(answer[0])

Answer the next question: In what country and what kind of institution did Whitfield work when he created Python? by reading the following text:As of early 2026, the latest major release is Python 3.13. Who Created It? Python was designed by the Dutch programmer Guido van Rossum. He began working on it in late 1989 at the CWI research institute in the Netherlands, during a Christmas holiday, as a personal project to keep himself busy during his time off.
